In [ ]:
# CP-style self-supervised training from a 5-channel NPY (inline Python, no shell commands)
from pathlib import Path
import os
import sys
import importlib
from types import SimpleNamespace
import numpy as np
import torch

In [ ]:
# Paths and run settings
REPO_DIR = Path("/Users/svo/source/dinov2")
NPY_PATH = Path("/scratch/svo/morpho_analysis/toy_dataset/MIPFTMA14/aveolar/processed_img_with_mask.npy")
CONFIG_FILE = REPO_DIR / "dinov2/configs/ssl_default_config.yaml"
OUTPUT_DIR = REPO_DIR / "outputs/cp_npy5_run1"

# Dataset/training settings
NUM_CHANNELS = 5
EPOCHS = 15
BATCH_SIZE_PER_GPU = 8
NUM_WORKERS = 32

# CP protocol knobs from the paper.
GLOBAL_CROP_SIZE = 128
LOCAL_CROPS_NUMBER = 0  # discard local crops for Cell Painting
EVAL_CENTER_CROP_SIZE = 128

assert REPO_DIR.exists(), f"Missing repo dir: {REPO_DIR}"
assert NPY_PATH.exists(), f"Missing dataset: {NPY_PATH}"
assert CONFIG_FILE.exists(), f"Missing config: {CONFIG_FILE}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

arr = np.load(NPY_PATH, mmap_mode="r")
print("Dataset shape:", arr.shape, "dtype:", arr.dtype, "min:", arr.min(), "max:", arr.max())
if arr.ndim != 4 or arr.shape[1] < NUM_CHANNELS:
    raise ValueError(f"Expected [N,C,H,W] with C>={NUM_CHANNELS}, got {arr.shape}")
if arr.shape[-1] < GLOBAL_CROP_SIZE or arr.shape[-2] < GLOBAL_CROP_SIZE:
    raise ValueError(
        f"Input spatial size must be >= {GLOBAL_CROP_SIZE} for CP protocol, got {arr.shape[-2:]}"
    )

OFFICIAL_EPOCH_LENGTH = int(np.ceil(arr.shape[0] / BATCH_SIZE_PER_GPU))
WARMUP_EPOCHS = max(0, min(8, EPOCHS - 1))
WARMUP_TEACHER_TEMP_EPOCHS = max(0, min(8, EPOCHS - 1))

print(
    f"Run settings -> N={arr.shape[0]}, channels={NUM_CHANNELS}, epochs={EPOCHS}, "
    f"epoch_len={OFFICIAL_EPOCH_LENGTH}, warmup={WARMUP_EPOCHS}, "
    f"global_crop={GLOBAL_CROP_SIZE}, local_crops={LOCAL_CROPS_NUMBER}, "
    f"eval_center_crop={EVAL_CENTER_CROP_SIZE}"
)

In [ ]:
# Runtime dataset + training patches (keeps repo files untouched)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import dinov2.train.train as train_mod
train_mod = importlib.reload(train_mod)
import dinov2.distributed as distributed
from torch.utils.data import Dataset
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP, StateDictType
from dinov2.fsdp import FSDPCheckpointer as OrigFSDPCheckpointer, rankstr

for key in [
    "SLURM_JOB_ID",
    "SLURM_JOB_NUM_NODES",
    "SLURM_JOB_NODELIST",
    "SLURM_PROCID",
    "SLURM_NTASKS",
    "SLURM_LOCALID",
]:
    os.environ.pop(key, None)

class NPYCPDataset(Dataset):
    def __init__(self, root, num_channels=5, transform=None, target_transform=None):
        self.arr = np.load(root, mmap_mode="r")
        self.num_channels = int(num_channels)
        self.transform = transform
        self.target_transform = target_transform

        if self.arr.ndim != 4:
            raise ValueError(f"Expected [N,C,H,W], got {self.arr.shape}")
        if self.arr.shape[1] < self.num_channels:
            raise ValueError(f"Expected at least {self.num_channels} channels, got {self.arr.shape[1]}")

    def __len__(self):
        return int(self.arr.shape[0])

    def __getitem__(self, idx):
        image = np.asarray(self.arr[idx, : self.num_channels]).astype(np.float32, copy=False)
        img_max = float(np.max(image)) if image.size > 0 else 0.0
        if img_max > 255.0:
            image = image * (255.0 / 65535.0)
        image = torch.from_numpy(image)
        target = 0

        if self.transform is not None:
            image = self.transform(image)
        if self.target_transform is not None:
            target = self.target_transform(target)
        return image, target

def _parse_dataset_str_runtime(dataset_str: str):
    tokens = dataset_str.split(":")
    name = tokens[0]
    kwargs = {}
    for token in tokens[1:]:
        k, v = token.split("=", 1)
        kwargs[k] = v
    return name, kwargs

if not hasattr(train_mod, "_orig_make_dataset_for_npycp"):
    train_mod._orig_make_dataset_for_npycp = train_mod.make_dataset

def _make_dataset_runtime(*, dataset_str, transform=None, target_transform=None):
    name, kwargs = _parse_dataset_str_runtime(dataset_str)
    if name == "NPYCP":
        root = kwargs["root"]
        num_channels = int(kwargs.get("wildcard", "5"))
        return NPYCPDataset(
            root=root,
            num_channels=num_channels,
            transform=transform,
            target_transform=target_transform,
        )
    return train_mod._orig_make_dataset_for_npycp(
        dataset_str=dataset_str,
        transform=transform,
        target_transform=target_transform,
    )

train_mod.make_dataset = _make_dataset_runtime

if not hasattr(train_mod, "_orig_collate_data_and_cast_for_npycp"):
    train_mod._orig_collate_data_and_cast_for_npycp = train_mod.collate_data_and_cast

def _collate_data_and_cast_fp32(*args, **kwargs):
    kwargs["dtype"] = torch.float32
    return train_mod._orig_collate_data_and_cast_for_npycp(*args, **kwargs)

train_mod.collate_data_and_cast = _collate_data_and_cast_fp32

class JupyterFSDPCheckpointer(OrigFSDPCheckpointer):
    def save(self, name: str, **kwargs):
        if not self.save_dir or not self.save_to_disk:
            return
        data = {}
        with FSDP.state_dict_type(self.model, StateDictType.FULL_STATE_DICT):
            data["model"] = self.model.state_dict()
        for key, obj in self.checkpointables.items():
            data[key] = obj.state_dict()
        data.update(kwargs)
        basename = f"{name}.{rankstr()}.pth"
        save_file = os.path.join(self.save_dir, basename)
        with self.path_manager.open(save_file, "wb") as f:
            torch.save(data, f)
        self.tag_last_checkpoint(basename)

    def load(self, *args, **kwargs):
        with FSDP.state_dict_type(self.model, StateDictType.FULL_STATE_DICT):
            return super().load(*args, **kwargs)

train_mod.FSDPCheckpointer = JupyterFSDPCheckpointer
print("Runtime patching complete: NPYCP dataset + FP32 collate + notebook-safe checkpointer")

In [ ]:
# Build training opts for CP-style (5-channel) self-supervised run
dataset_opt = f"train.dataset_path=NPYCP:root={NPY_PATH}:wildcard={NUM_CHANNELS}"
opts = [
    dataset_opt,
    f"train.num_workers={NUM_WORKERS}",
    f"train.batch_size_per_gpu={BATCH_SIZE_PER_GPU}",
    f"train.OFFICIAL_EPOCH_LENGTH={OFFICIAL_EPOCH_LENGTH}",
    "train.cell_augmentation=true",
    f"optim.epochs={EPOCHS}",
    f"optim.warmup_epochs={WARMUP_EPOCHS}",
    f"teacher.warmup_teacher_temp_epochs={WARMUP_TEACHER_TEMP_EPOCHS}",
    "evaluation.eval_period_iterations=0",
    "student.arch=vit_small",
    "student.patch_size=8",
    f"student.in_chans={NUM_CHANNELS}",
    f"teacher.in_chans={NUM_CHANNELS}",
    f"crops.global_crops_size={GLOBAL_CROP_SIZE}",
    f"crops.local_crops_number={LOCAL_CROPS_NUMBER}",
    # Full precision for 5-channel stability in notebook runs.
    "compute_precision.grad_scaler=false",
    "compute_precision.student.backbone.mixed_precision.param_dtype=fp32",
    "compute_precision.student.backbone.mixed_precision.reduce_dtype=fp32",
    "compute_precision.student.backbone.mixed_precision.buffer_dtype=fp32",
    "compute_precision.teacher.backbone.mixed_precision.param_dtype=fp32",
    "compute_precision.teacher.backbone.mixed_precision.reduce_dtype=fp32",
    "compute_precision.teacher.backbone.mixed_precision.buffer_dtype=fp32",
]

for item in opts:
    print(item)

In [ ]:
# Start training (inline, no shell command)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this training run.")
if distributed.is_enabled():
    raise RuntimeError(
        "Distributed process group is already initialized in this kernel. "
        "Restart kernel before running this cell again."
    )

args = SimpleNamespace(
    config_file=str(CONFIG_FILE),
    no_resume=True,
    eval_only=False,
    eval="",
    opts=list(opts),
    output_dir=str(OUTPUT_DIR),
    seed=0,
)

cfg = train_mod.setup(args)
model = train_mod.SSLMetaArch(cfg).to(torch.device("cuda"))
model.prepare_for_distributed_training()
train_mod.do_train(cfg, model, resume=not args.no_resume)

print(f"Training completed. Outputs saved to: {OUTPUT_DIR}")

In [ ]:
# Optional: inspect produced checkpoints
eval_ckpts = sorted((OUTPUT_DIR / "eval").glob("training_*/teacher_checkpoint.pth")) if (OUTPUT_DIR / "eval").exists() else []
model_ckpts = sorted(OUTPUT_DIR.glob("model_*.rank_0.pth"))

print("Teacher eval checkpoints:", len(eval_ckpts))
if eval_ckpts:
    print("Latest teacher checkpoint:", eval_ckpts[-1])
print("Model checkpoints:", len(model_ckpts))
if model_ckpts:
    print("Latest model checkpoint:", model_ckpts[-1])